# Single-Agent Pipeline Project

A small, dependency-free implementation of a **stateful directed graph** agent pipeline, demonstrating:

- Stateful directed graph execution (nodes + edges + shared state)
- Conditional routing based on query intent
- Retry loops for unreliable tool calls
- JSON-schema-validated tool inputs/outputs
- Error handling (try/except + retries + logging)
- Trajectory logging (every step, not just the final answer)
- Evaluation metrics: task completion rate + cost (tool calls / time)

No external packages required — runs with just the Python standard library.

## 1. Imports

In [1]:
from __future__ import annotations

import random
import re
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional


## 2. State object

The `AgentState` is shared, mutable, and carried across every node in the graph. Every node reads from and writes to it — this is what lets the graph *remember* previous steps instead of behaving like a stateless, fixed-order linear pipeline.

In [2]:
@dataclass
class AgentState:
    """The 'state' in the stateful directed graph."""
    query: str
    trajectory: List[Dict[str, Any]] = field(default_factory=list)
    tool_calls: int = 0
    retries: int = 0
    result: Optional[Any] = None
    error: Optional[str] = None
    route: Optional[str] = None
    done: bool = False

    def log_step(self, node: str, detail: str, **extra: Any) -> None:
        self.trajectory.append({
            "node": node,
            "detail": detail,
            **extra,
        })


## 3. Minimal JSON-schema validation

A small hand-rolled JSON-schema-style validator (subset: `type`, `required`, `enum`). Real projects would use the `jsonschema` package — this keeps the notebook dependency-free while showing the same idea: tool inputs/outputs must conform to a declared structure before they're trusted by the rest of the pipeline.

In [3]:
class SchemaError(Exception):
    """Raised when data does not conform to a tool's declared schema."""


def validate_schema(data: Dict[str, Any], schema: Dict[str, Any]) -> None:
    if schema.get("type") == "object":
        if not isinstance(data, dict):
            raise SchemaError(f"Expected object, got {type(data).__name__}")
        for req in schema.get("required", []):
            if req not in data:
                raise SchemaError(f"Missing required field: '{req}'")
        for key, value in data.items():
            prop_schema = schema.get("properties", {}).get(key)
            if not prop_schema:
                continue
            expected_type = prop_schema.get("type")
            type_map = {"string": str, "number": (int, float), "boolean": bool}
            if expected_type in type_map and not isinstance(value, type_map[expected_type]):
                raise SchemaError(
                    f"Field '{key}' expected {expected_type}, got {type(value).__name__}"
                )
            if "enum" in prop_schema and value not in prop_schema["enum"]:
                raise SchemaError(f"Field '{key}'={value!r} not in enum {prop_schema['enum']}")


## 4. Tools

Each tool declares an input schema and an output schema, and is validated against both. The calculator tool also simulates occasional transient failures so the retry loop has something to do.

In [4]:
CALCULATOR_INPUT_SCHEMA = {
    "type": "object",
    "required": ["expression"],
    "properties": {"expression": {"type": "string"}},
}
CALCULATOR_OUTPUT_SCHEMA = {
    "type": "object",
    "required": ["value"],
    "properties": {"value": {"type": "number"}},
}

KEYWORD_INPUT_SCHEMA = {
    "type": "object",
    "required": ["text"],
    "properties": {"text": {"type": "string"}},
}
KEYWORD_OUTPUT_SCHEMA = {
    "type": "object",
    "required": ["keywords"],
    "properties": {"keywords": {"type": "string"}},
}


def calculator_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Evaluates a simple arithmetic expression. Simulates occasional
    transient failures so the retry loop has something to do."""
    validate_schema(payload, CALCULATOR_INPUT_SCHEMA)

    if random.random() < 0.3:
        raise ConnectionError("Simulated transient calculator API failure")

    expr = payload["expression"]
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expr):
        raise ValueError(f"Unsafe or invalid expression: {expr!r}")

    value = eval(expr, {"__builtins__": {}}, {})  # sandboxed chars only
    output = {"value": float(value)}
    validate_schema(output, CALCULATOR_OUTPUT_SCHEMA)
    return output


def keyword_extraction_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Extracts naive 'keywords' (words longer than 4 chars, deduped)."""
    validate_schema(payload, KEYWORD_INPUT_SCHEMA)

    words = re.findall(r"[a-zA-Z]{5,}", payload["text"].lower())
    keywords = sorted(set(words))
    output = {"keywords": ", ".join(keywords) if keywords else "(none found)"}
    validate_schema(output, KEYWORD_OUTPUT_SCHEMA)
    return output


def general_response_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Fallback module for anything that isn't calculator/keyword intent."""
    return {"response": f"I don't have a specialized tool for this, but here's "
                         f"my best general answer to: {payload['text']!r}"}


## 5. Graph nodes

Each node is a function `(state) -> state`. `node_query_analysis` performs conditional routing: it inspects the query and decides which tool node to visit next. The tool nodes wrap their tool call with a retry loop and error handling.

In [5]:
def node_query_analysis(state: AgentState) -> AgentState:
    """Conditional routing: decide which node to visit next based on
    simple rule-based intent detection over the query text."""
    q = state.query.lower()
    if "calculate" in q or re.search(r"[0-9].*[+\-*/].*[0-9]", q):
        state.route = "calculator"
    elif "keyword" in q:
        state.route = "keywords"
    else:
        state.route = "general"

    state.log_step("query_analysis", f"Routed query to '{state.route}'", route=state.route)
    return state


def node_calculator(state: AgentState) -> AgentState:
    """Wraps calculator_tool with a retry loop and error handling."""
    max_retries = 3
    match = re.search(r"[-+]?\d*\.?\d+(?:\s*[+\-*/]\s*[-+]?\d*\.?\d+)+", state.query)
    expression = match.group(0) if match else "0"

    for attempt in range(1, max_retries + 1):
        state.tool_calls += 1
        try:
            output = calculator_tool({"expression": expression})
            state.result = output["value"]
            state.log_step(
                "calculator_tool",
                f"Computed {expression} = {output['value']} (attempt {attempt})",
                attempt=attempt,
            )
            return state
        except (ConnectionError, SchemaError, ValueError) as exc:
            state.retries += 1
            state.log_step(
                "calculator_tool",
                f"Attempt {attempt} failed: {exc}",
                attempt=attempt,
                error=str(exc),
            )
            time.sleep(0.01)  # brief backoff, kept tiny for demo speed

    state.error = f"Calculator tool failed after {max_retries} attempts"
    state.log_step("calculator_tool", state.error)
    return state


def node_keyword_extraction(state: AgentState) -> AgentState:
    state.tool_calls += 1
    try:
        output = keyword_extraction_tool({"text": state.query})
        state.result = output["keywords"]
        state.log_step("keyword_extraction_tool", f"Extracted keywords: {output['keywords']}")
    except SchemaError as exc:
        state.error = str(exc)
        state.log_step("keyword_extraction_tool", f"Failed: {exc}", error=str(exc))
    return state


def node_general_response(state: AgentState) -> AgentState:
    state.tool_calls += 1
    output = general_response_tool({"text": state.query})
    state.result = output["response"]
    state.log_step("general_response", output["response"])
    return state


def node_finalize(state: AgentState) -> AgentState:
    state.done = True
    status = "FAILED" if state.error else "SUCCESS"
    state.log_step("finalize", f"Pipeline finished with status: {status}")
    return state


## 6. The graph

`AgentGraph` is a minimal stateful directed graph executor:

- `nodes` maps node name → function(state) → state
- `edges` maps node name → either the next node name, or a callable that inspects `state` and returns the next node name (a **conditional edge**)

In [6]:
class AgentGraph:
    def __init__(self) -> None:
        self.nodes: Dict[str, Callable[[AgentState], AgentState]] = {}
        self.edges: Dict[str, Callable[[AgentState], str]] = {}
        self.entry_point: Optional[str] = None

    def add_node(self, name: str, fn: Callable[[AgentState], AgentState]) -> "AgentGraph":
        self.nodes[name] = fn
        return self

    def add_edge(self, from_node: str, to_node: str) -> "AgentGraph":
        self.edges[from_node] = lambda state: to_node
        return self

    def add_conditional_edge(
        self, from_node: str, router: Callable[[AgentState], str]
    ) -> "AgentGraph":
        self.edges[from_node] = router
        return self

    def set_entry_point(self, name: str) -> "AgentGraph":
        self.entry_point = name
        return self

    def run(self, query: str, max_steps: int = 20) -> AgentState:
        if not self.entry_point:
            raise RuntimeError("Graph has no entry point set")

        state = AgentState(query=query)
        current = self.entry_point
        steps = 0

        while current and steps < max_steps:
            fn = self.nodes.get(current)
            if fn is None:
                raise RuntimeError(f"Unknown node: {current}")
            state = fn(state)
            steps += 1

            if state.done:
                break

            edge_fn = self.edges.get(current)
            current = edge_fn(state) if edge_fn else None

        return state


## 7. Build the pipeline

Wire the nodes together: `query_analysis` routes conditionally to `calculator`, `keywords`, or `general`, and all three converge on `finalize`.

In [7]:
def build_pipeline() -> AgentGraph:
    graph = AgentGraph()
    graph.add_node("query_analysis", node_query_analysis)
    graph.add_node("calculator", node_calculator)
    graph.add_node("keywords", node_keyword_extraction)
    graph.add_node("general", node_general_response)
    graph.add_node("finalize", node_finalize)

    graph.set_entry_point("query_analysis")

    # Conditional edge: route decided in node_query_analysis
    graph.add_conditional_edge("query_analysis", lambda s: s.route)

    # All tool nodes converge on finalize
    graph.add_edge("calculator", "finalize")
    graph.add_edge("keywords", "finalize")
    graph.add_edge("general", "finalize")

    return graph


## 8. Evaluation harness

Runs a batch of queries through the graph, prints the full trajectory for each one, and reports **task completion rate** and **cost** (tool calls, retries, wall-clock time).

In [8]:
@dataclass
class EvalSummary:
    total_tasks: int
    completed_tasks: int
    total_tool_calls: int
    total_retries: int
    total_time_s: float

    @property
    def completion_rate(self) -> float:
        return self.completed_tasks / self.total_tasks if self.total_tasks else 0.0

    @property
    def avg_cost_per_task(self) -> float:
        return self.total_tool_calls / self.total_tasks if self.total_tasks else 0.0

    def report(self) -> str:
        return (
            f"Task completion rate : {self.completion_rate:.0%} "
            f"({self.completed_tasks}/{self.total_tasks})\n"
            f"Total tool calls      : {self.total_tool_calls}\n"
            f"Total retries         : {self.total_retries}\n"
            f"Avg tool calls/task   : {self.avg_cost_per_task:.2f}\n"
            f"Total wall-clock time : {self.total_time_s:.3f}s"
        )


def run_eval(graph: AgentGraph, queries: List[str]) -> EvalSummary:
    completed = 0
    total_calls = 0
    total_retries = 0
    start = time.time()

    for q in queries:
        run_id = uuid.uuid4().hex[:8]
        print(f"\n=== Run {run_id} ===")
        print(f"Query: {q!r}")

        state = graph.run(q)

        total_calls += state.tool_calls
        total_retries += state.retries
        if not state.error:
            completed += 1

        print("Trajectory:")
        for i, step in enumerate(state.trajectory, 1):
            print(f"  {i}. [{step['node']}] {step['detail']}")

        print(f"Result: {state.result!r}" + (f"  (ERROR: {state.error})" if state.error else ""))

    elapsed = time.time() - start
    return EvalSummary(
        total_tasks=len(queries),
        completed_tasks=completed,
        total_tool_calls=total_calls,
        total_retries=total_retries,
        total_time_s=elapsed,
    )


## 9. Demo run

Run 4 example queries through the pipeline and print the evaluation summary.

In [9]:
random.seed(7)  # reproducible demo output

pipeline = build_pipeline()

demo_queries = [
    "please calculate 12 + 8 * 2 for me",
    "extract the keywords from this sentence about artificial intelligence agents",
    "what's the weather like today?",
    "calculate 100 / 4 - 5",
]

summary = run_eval(pipeline, demo_queries)

print("\n" + "=" * 40)
print("EVALUATION SUMMARY")
print("=" * 40)
print(summary.report())



=== Run f0d5ea96 ===
Query: 'please calculate 12 + 8 * 2 for me'
Trajectory:
  1. [query_analysis] Routed query to 'calculator'
  2. [calculator_tool] Computed 12 + 8 * 2 = 28.0 (attempt 1)
  3. [finalize] Pipeline finished with status: SUCCESS
Result: 28.0

=== Run 32fb0106 ===
Query: 'extract the keywords from this sentence about artificial intelligence agents'
Trajectory:
  1. [query_analysis] Routed query to 'keywords'
  2. [keyword_extraction_tool] Extracted keywords: about, agents, artificial, extract, intelligence, keywords, sentence
  3. [finalize] Pipeline finished with status: SUCCESS
Result: 'about, agents, artificial, extract, intelligence, keywords, sentence'

=== Run 26cfdb68 ===
Query: "what's the weather like today?"
Trajectory:
  1. [query_analysis] Routed query to 'general'
  2. [general_response] I don't have a specialized tool for this, but here's my best general answer to: "what's the weather like today?"
  3. [finalize] Pipeline finished with status: SUCCESS
Resu

## 10. Try your own query

Edit the `query` variable below and re-run the cell to see the pipeline route and answer a question of your own.

In [10]:
query = "calculate 45 * 3 + 10"

state = pipeline.run(query)
print(f"Query: {query!r}\n")
print("Trajectory:")
for i, step in enumerate(state.trajectory, 1):
    print(f"  {i}. [{step['node']}] {step['detail']}")
print(f"\nResult: {state.result!r}")


Query: 'calculate 45 * 3 + 10'

Trajectory:
  1. [query_analysis] Routed query to 'calculator'
  2. [calculator_tool] Attempt 1 failed: Simulated transient calculator API failure
  3. [calculator_tool] Computed 45 * 3 + 10 = 145.0 (attempt 2)
  4. [finalize] Pipeline finished with status: SUCCESS

Result: 145.0
